### Reading the file



In [ ]:
import pandas as pd
df = pd.read_csv('../Data/SBAnational.csv',on_bad_lines='skip')

In [ ]:
df.describe()

In [ ]:
df.info(verbose=True,show_counts=True)

### Split as Target and Input

In [ ]:
Y=df['MIS_Status']
df.drop(columns=['MIS_Status'],inplace=True)   



In [ ]:
df.isnull().mean().sort_values(ascending=False)

Check if final data is imbalanced. Apply SMOTE is ratio 20:1

In [ ]:
Y.value_counts()

### Dropping more columns because they are causing data leakage

In [ ]:
X=df.drop(columns=['ChgOffPrinGr','BalanceGross','ChgOffDate'])


### Train Test splitting

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

### Split your Xtrain data to two different types of categories. Numerical vs Categorical. You further have to split Categorical as Nominal and Ordinal

In [ ]:
numCols=list(X_train.select_dtypes(include=['int64','float64']).columns)
catCols=list(X_train.select_dtypes(include=['object']).columns)

In [ ]:
numCols



### Sending correct column to correct array

In [ ]:
wrongCat=['DisbursementGross','GrAppv','SBA_Appv','DisbursementDate','ApprovalDate','ApprovalFY']
wrongNum=['NewExist','UrbanRural']

### Removing wrong data and appending it correctly to numCols

In [ ]:
for col in wrongCat:
    catCols.remove(col)
numCols.extend(wrongCat)



### Removing wrong data from numCols and appending it correctly to catCols

In [ ]:
for cols in wrongNum:
    numCols.remove(cols)
catCols.extend(wrongNum)

In [ ]:
catCols

## Imputing missing values

### Numerical

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

In [ ]:
df[numCols].isnull().mean().sort_values(ascending=False)

Got concluded that there is no missing values in numerical columns

In [ ]:
numImpute=SimpleImputer(strategy='mean')
colScale=StandardScaler()
numPipeline = Pipeline(steps=[
    ('numImp',numImpute),
    ('numScaling',colScale)
])

### Categorical

In [ ]:
df[catCols].isnull().mean().sort_values(ascending=False)

### Split the Categorical into Nominal and Ordinal before encoding

In [ ]:
X_train[numCols].dtypes


There are no ordinal columns so no need to apply ordinal encoding. Only apply one hot encoder

There are missing values in categorical hence impute using strategy=most-frequent

In [ ]:
ohe=OneHotEncoder()

In [ ]:
catImpute=SimpleImputer(strategy='most-frequent')
catPipeline=Pipeline(steps=[
    ('catImp',catImpute),
    ('catEncoding',ohe),
    
])



Need to parse the data in numerical column

In [ ]:
df['ApprovalDate']=pd.to_datetime(df['ApprovalDate'],format='%d-%b-%y', errors='coerce')
df['ApprovalMonth']=df['ApprovalDate'].dt.month
df['ApprovalYear']=df['ApprovalDate'].dt.year
df['ApprovalDay']=df['ApprovalDate'].dt.day

In [ ]:
df['DisbursementDate']=pd.to_datetime(df['DisbursementDate'],format='%d-%b-%y', errors='coerce')
df['DisbursementMonth']=df['DisbursementDate'].dt.month
df['DisbursementYear']=df['DisbursementDate'].dt.year
df['DisbursementDay']=df['DisbursementDate'].dt.day

df['ApprovalYear'] = pd.to_numeric(df['ApprovalFY'].astype(str).str.extract(r'(\d+)')[0], errors='coerce')


In [ ]:
removeDate=['DisbursementDate','ApprovalDate','ApprovalFY']
for date in removeDate:
    numCols.remove(date)

dateCols=['DisbursementMonth','DisbursementYear','DisbursementDay','ApprovalYear','ApprovalMonth','ApprovalDay','ApprovalFY']
numCols.extend(dateCols)


In [ ]:
df['GrAppv']=pd.to_numeric(df['GrAppv'].astype(str).str.replace(r'[\ $ ,]', '' ),errors='coerce')
df['DisbursementGross']=pd.to_numeric(df['DisbursementGross'].astype(str).str.replace(r'[$ , \]', ''), errors='coerce')

In [ ]:
df[catCols].dtypes

### Transformer

In [ ]:
from sklearn.compose import ColumnTransformer

In [ ]:
processed=ColumnTransformer(transformers=[
    ('numProcessed',numPipeline,numCols),
    ('catProcessed',catPipeline,catCols)
])

In [ ]:
print("DisbursementMonth" in X_train.columns) 

In [ ]:
X_train_processed=processed.fit_transform(X_train)
X_test_processed=processed.transform(X_test)